# Track 1 · 06 Signal — is the image information there at all?

Ten model configurations have landed within noise of chance: ridge, a ClipLoss probe, and
eight LoRA variants spanning three orders of magnitude of effective capacity. Before tuning
an eleventh, this asks whether the thing we are trying to decode is present in the data and
where in time it sits.

**No REVE here.** Ridge straight from raw EEG samples. If the raw signal cannot do it
either, the encoder is exonerated and the problem is upstream of every model we have tried.

---

## The three questions

1. **Does the scan work at all?** A positive control: plant a known signal at a known
   latency and confirm the scan finds it. Without this, a flat result is uninterpretable,
   because "no signal" and "broken detector" look identical.
2. **Is the canonical window decodable?** Ridge from the raw window the track config
   specifies, against a permutation null.
3. **Where is the response in time?** Slide a short window from -300 to +700 ms. A visual
   response should show up somewhere after 0. If the peak sits at a positive offset, the
   marker stream lags the stimulus and every epoch we have built has been on the wrong
   part of the trace.

Question 3 is the one with a specific suspect. Alljoined's Emotiv marker stream is the same
one with the documented stalling bug that made `neuralfetch` abort, and subject decoding
works fine at 62.5% while image decoding does not. Subject identity needs no temporal
precision; a visual evoked response needs tens of milliseconds.

## Why a short sliding window

The track config's window is 1.0 s, but the ISI is ~215 ms, so it spans about five
stimuli and smears the response of the one we are labelling. For *locating* a response a
short window is the right instrument, and 300 ms at 200 Hz is 60 samples per channel.

Everything is epoched once over a wide window and then sliced, so the scan costs one pass
over the EEG rather than one per offset.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

SFREQ = 200.0
WIDE = (-0.4, 0.9)          # epoch once over this, then slice
SCAN_LEN = 0.30             # sliding window length, seconds
SCAN_FROM, SCAN_TO, SCAN_STEP = -0.30, 0.60, 0.05
DECIMATE = 2                # for the full-window test: 200 -> 100 Hz keeps ERP content

PLANT_LATENCY = 0.15        # positive control: inject a response at +150 ms
PLANT_SNR = 0.30            # amplitude relative to the data's own sd

N_PERM = 200
VAL_IMAGE_FRAC = 0.2
SEED = 33

In [ ]:
%pip install -q mne

import importlib
for mod in ('mne', 'numpy'):
    print(f'{mod:8} {getattr(importlib.import_module(mod), "__version__", "ok")}')

In [ ]:
import json, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, mne
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
PREP, RAW = BASE / 'prepared', BASE / 'raw'
DATASET = 'nm000134'
STIM = {'train': 'stim_train', 'test': 'stim_test'}
rng = np.random.default_rng(SEED)


def l2(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def rank_pct(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    tc = sims[np.arange(len(sims)), truth]
    return float((sims < tc[:, None]).mean())


def top5(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    k = min(5, len(gal))
    tk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, tk)]), 100.0 * k / len(gal)


# Capped deliberately. p > n here, which ridge handles, and a grid reaching 1e5
# lets the fit collapse to a constant -- at which point every offset returns the
# same number and the scan is blind. Selection also happens once, not per offset,
# so offsets differ only by the data in them.
LAMBDAS = np.logspace(-3, 1, 9)


def ridge_predict(Xa, Ya, Xb, Xv=None, yv=None, emb=None, lam_rel=None):
    """Fit on (Xa, Ya). Returns (predictions for Xb, lambda used relative to the
    mean eigenvalue). Pass lam_rel to reuse a previously chosen value."""
    mu, sd = Xa.mean(0), Xa.std(0) + 1e-8
    Xac = (Xa - mu) / sd
    ym = Ya.mean(0)
    G = Xac.T @ Xac
    scale = np.trace(G) / G.shape[0]
    d, V = np.linalg.eigh(G)
    VtA = V.T @ (Xac.T @ (Ya - ym))
    if lam_rel is None:
        if Xv is None:
            lam_rel = 1.0
        else:
            Xvc = (Xv - mu) / sd
            lam_rel = max(
                ((rank_pct(Xvc @ (V @ (VtA / (d + l * scale)[:, None])) + ym, yv, emb), l)
                 for l in LAMBDAS))[1]
    W = V @ (VtA / (d + lam_rel * scale)[:, None])
    return ((Xb - mu) / sd) @ W + ym, lam_rel


def spread_of(pred):
    """Relative variation across windows. Near zero means the fit collapsed to a
    constant, which makes rank_pct permutation-invariant and the null degenerate."""
    return float(np.linalg.norm(pred - pred.mean(0), axis=1).mean()
                 / (np.linalg.norm(pred.mean(0)) + 1e-12))


print('helpers ready')

## Step 1 — epoch once, wide

One pass over the EEG at `-0.4 … 0.9 s`. Every later test slices this array, so no
recording is read twice. Preprocessing matches what `track1_02_features` recorded, read
from the feature bundle so the two cannot drift apart.

In [ ]:
f = np.load(PREP / 'features_pooled.npz', allow_pickle=True)
t = np.load(PREP / 'image_embeddings.npz', allow_pickle=True)
emb = l2(t['emb'].astype('float32'))
key = {k: i for i, k in enumerate(t['keys'].tolist())}
man = json.loads((PREP / 'prepare_manifest.json').read_text())
PP = dict(filt=[float(x) for x in np.atleast_1d(f['filt'])] if f['filt'].size else None,
          scale=str(f['scale_scope']), clamp=float(f['clamp']))
print('preprocessing from the bundle:', PP, '\n')

lo, hi = int(round(WIDE[0] * SFREQ)), int(round(WIDE[1] * SFREQ))
NW = hi - lo
T0 = -lo                                   # index of t=0 inside each window


def wide_windows(rel, split):
    raw = mne.io.read_raw_edf(RAW / DATASET / rel, preload=True)
    if PP['filt']:
        raw.filter(PP['filt'][0], PP['filt'][1], verbose=False)
    raw.resample(SFREQ)
    d = raw.get_data().astype('float32')
    tsv = RAW / DATASET / rel.replace('_eeg.edf', '_events.tsv')
    df = pd.read_csv(tsv, sep='\t')
    df = df[df.trial_type == STIM[split]].dropna(subset=['stim_file']).reset_index(drop=True)
    s = np.round(df.onset.values * SFREQ).astype(int)
    ok = (s + lo >= 0) & (s + hi < d.shape[1])
    df, s = df[ok].reset_index(drop=True), s[ok]
    X = np.stack([d[:, i + lo:i + hi] for i in s])
    return X, df.stim_file.values


t0 = time.time()
parts = {'train': [[], [], []], 'test': [[], [], []]}
for rel, meta in sorted(man['recordings'].items()):
    sp = meta['split']
    X, stim = wide_windows(rel, sp)
    idx = np.array([key.get(s, -1) for s in stim])
    k = idx >= 0
    parts[sp][0].append(X[k]); parts[sp][1].append(idx[k])
    parts[sp][2].append(np.full(int(k.sum()), meta['subject']))
Wtr = [np.concatenate(a) for a in parts['train']]
Wte = [np.concatenate(a) for a in parts['test']]
CH = mne.io.read_raw_edf(RAW / DATASET / sorted(man['recordings'])[0], preload=False).ch_names
print(f'train {Wtr[0].shape}   test {Wte[0].shape}   ({time.time() - t0:.0f}s, '
      f'{(Wtr[0].nbytes + Wte[0].nbytes) / 1e9:.2f} GB)')

val_imgs = set(rng.choice(np.unique(Wtr[1]),
                          int(len(np.unique(Wtr[1])) * VAL_IMAGE_FRAC),
                          replace=False).tolist())
IV = np.array([y in val_imgs for y in Wtr[1]])
print(f'{(~IV).sum():,} fit / {IV.sum():,} val windows')

In [ ]:
def slice_feats(X, start_s, length_s, decimate=1):
    """Cut [start, start+length) relative to stimulus onset, standardise per window
    per channel the way the features notebook did, clamp, and flatten."""
    a = T0 + int(round(start_s * SFREQ))
    b = a + int(round(length_s * SFREQ))
    Z = X[:, :, a:b]
    if decimate > 1:
        n = (Z.shape[2] // decimate) * decimate
        Z = Z[:, :, :n].reshape(Z.shape[0], Z.shape[1], -1, decimate).mean(3)
    if PP['scale'] == 'window':
        Z = (Z - Z.mean(2, keepdims=True)) / (Z.std(2, keepdims=True) + 1e-8)
    Z = np.clip(Z, -PP['clamp'], PP['clamp'])
    return Z.reshape(len(Z), -1).astype('float64')


def evaluate(start_s, length_s, Xtr_raw, Xte_raw, decimate=1, lam_rel=None):
    A = slice_feats(Xtr_raw, start_s, length_s, decimate)
    B = slice_feats(Xte_raw, start_s, length_s, decimate)
    pred, lam = ridge_predict(A[~IV], emb[Wtr[1][~IV]], B,
                              A[IV], Wtr[1][IV], emb, lam_rel)
    return rank_pct(pred, Wte[1], emb), top5(pred, Wte[1], emb)[0], spread_of(pred), lam


print(f'scan window {SCAN_LEN * 1000:.0f} ms -> '
      f'{int(SCAN_LEN * SFREQ)} samples x {len(CH)} ch = '
      f'{int(SCAN_LEN * SFREQ) * len(CH)} features')

## Step 2 — positive control

Inject a synthetic response at a known latency into a *copy* of the data, then run the same
scan. If the scan cannot find a planted signal, it cannot be trusted to report the absence
of a real one.

The plant is a target-derived pattern added at `PLANT_LATENCY`, at `PLANT_SNR` times the
data's own standard deviation, so it is weak on purpose. Nothing here touches the arrays
used in Steps 3 and 4.

In [ ]:
def plant(X, y_ids, latency, snr, proj):
    """Add a fixed linear projection of the target embedding at one latency."""
    Z = X.copy()
    a = T0 + int(round(latency * SFREQ))
    b = a + int(round(0.10 * SFREQ))               # 100 ms of injected response
    pat = (emb[y_ids] @ proj).astype('float32')    # (n, chans)
    pat /= pat.std() + 1e-12                       # or SNR means 1/sqrt(dim) of what
                                                   # it says, and the control is blind
    bump = np.hanning(b - a).astype('float32')
    Z[:, :, a:b] += Z.std() * snr * pat[:, :, None] * bump[None, None, :]
    return Z


proj = rng.standard_normal((emb.shape[1], len(CH))).astype('float32') / np.sqrt(emb.shape[1])
Ptr = plant(Wtr[0], Wtr[1], PLANT_LATENCY, PLANT_SNR, proj)
Pte = plant(Wte[0], Wte[1], PLANT_LATENCY, PLANT_SNR, proj)

offsets = np.arange(SCAN_FROM, SCAN_TO + 1e-9, SCAN_STEP)
_, _, _, LAM = evaluate(0.0, SCAN_LEN, Ptr, Pte)   # choose lambda once, reuse below
ctrl = np.array([evaluate(o, SCAN_LEN, Ptr, Pte, lam_rel=LAM)[0] for o in offsets])

# Report window CENTRE, not start: a response at latency L is best captured by a
# window centred on L, so plotting the start shifts the apparent peak by half the
# window length.
centres = offsets + SCAN_LEN / 2
peak = centres[ctrl.argmax()]
print(f'{"centre (ms)":>12}  rank')
for cm, r in zip(centres, ctrl):
    bar = '#' * max(0, int((r - 0.5) * 300))
    print(f'{cm * 1000:12.0f}  {r:.4f} {bar}{"   <- peak" if cm == peak else ""}')
plant_centre = PLANT_LATENCY + 0.05          # the injected bump is 100 ms wide
print(f'\nplanted at {plant_centre * 1000:.0f} ms (centre), scan peaks at '
      f'{peak * 1000:.0f} ms')
ok = abs(peak - plant_centre) <= SCAN_LEN / 2
print('CONTROL PASSES: the scan localises a planted signal, so a flat result below'
      if ok else 'CONTROL FAILS: the scan cannot find a known signal. Fix this first;')
print('means no signal rather than no detector.' if ok else
      'nothing below is interpretable.')
del Ptr, Pte

## Step 3 — is the canonical window decodable from raw EEG?

The exact window the track config specifies, straight from samples, with a permutation null
so "close to 0.5" is judged against its own noise rather than by eye.

In [ ]:
t0 = time.time()
A = slice_feats(Wtr[0], -0.2, 1.0, DECIMATE)
B = slice_feats(Wte[0], -0.2, 1.0, DECIMATE)
print(f'features {A.shape[1]} per window ({len(CH)} ch x {A.shape[1] // len(CH)} samples '
      f'at {SFREQ / DECIMATE:.0f} Hz)')
pred, lam_full = ridge_predict(A[~IV], emb[Wtr[1][~IV]], B,
                               A[IV], Wtr[1][IV], emb)
r_obs = rank_pct(pred, Wte[1], emb)
acc, chance = top5(pred, Wte[1], emb)

gal = np.unique(Wte[1])
pos = {g: i for i, g in enumerate(gal)}
truth = np.array([pos[y] for y in Wte[1]])
G = l2(emb[gal])
sims = l2(pred) @ G.T


def rank_of(perm):
    tc = sims[np.arange(len(sims)), perm]
    return float((sims < tc[:, None]).mean())


null = np.array([rank_of(rng.permutation(truth)) for _ in range(N_PERM)])
p = float((null >= r_obs).mean())
print(f'\nraw EEG, canonical window   rank {r_obs:.4f}   top-5 {acc:.2f}% '
      f'(chance {chance:.2f}%)')
print(f'null over {N_PERM} permutations  mean {null.mean():.4f}  sd {null.std():.4f}  '
      f'95th {np.percentile(null, 95):.4f}')
print(f'p = {p:.3f}   lambda {lam_full:.3g} x mean eigenvalue   ({time.time() - t0:.0f}s)')
sp = spread_of(pred)
print(f'prediction spread {sp:.4f}')
if sp < 1e-3 or null.std() < 1e-6:
    print('  WARNING: predictions are constant, so rank_pct is permutation-invariant')
    print('  and this null carries no information. Lower the lambda grid.')
print(f'\nfor comparison, REVE pooled features gave rank 0.4998, top-5 2.48%')

## Step 4 — where is the response in time?

In [ ]:
t0 = time.time()
_, _, _, LAM_R = evaluate(0.0, SCAN_LEN, Wtr[0], Wte[0])
real = [(o,) + evaluate(o, SCAN_LEN, Wtr[0], Wte[0], lam_rel=LAM_R)[:3] for o in offsets]

print(f'{"centre (ms)":>12}  {"rank":>7}  {"top-5":>6}  {"spread":>7}')
best = max(real, key=lambda x: x[1])
for o, r, a, sp_ in real:
    bar = '#' * max(0, int((r - 0.5) * 300))
    print(f'{(o + SCAN_LEN / 2) * 1000:12.0f}  {r:7.4f}  {a:5.2f}%  {sp_:7.4f} {bar}'
          f'{"   <- peak" if o == best[0] else ""}')
best_centre = best[0] + SCAN_LEN / 2
print(f'\npeak at {best_centre * 1000:.0f} ms (centre): rank {best[1]:.4f}, '
      f'top-5 {best[2]:.2f}%')
print(f'control peaked at {peak * 1000:.0f} ms for a signal planted at '
      f'{plant_centre * 1000:.0f} ms   ({time.time() - t0:.0f}s)')

# The Step 3 null came from a 3,200-feature fit; this scan uses 1,920, so give the
# peak its own null rather than borrowing one at a different dimensionality.
Ab = slice_feats(Wtr[0], best[0], SCAN_LEN)
Bb = slice_feats(Wte[0], best[0], SCAN_LEN)
pb, _ = ridge_predict(Ab[~IV], emb[Wtr[1][~IV]], Bb, lam_rel=LAM_R)
simsb = l2(pb) @ G.T
nullb = np.array([float((simsb < simsb[np.arange(len(simsb)),
                                        rng.permutation(truth)][:, None]).mean())
                  for _ in range(N_PERM)])
pval = float((nullb >= best[1]).mean())
print(f'\npeak null over {N_PERM} permutations  mean {nullb.mean():.4f}  '
      f'sd {nullb.std():.4f}  95th {np.percentile(nullb, 95):.4f}   p = {pval:.3f}')

print()
if pval >= 0.05:
    print('Flat. No window from -300 to +600 ms decodes above its own noise, and the')
    print('control shows the scan would have found a signal if one were there. The')
    print('image information is not recoverable from this recording at this SNR, which')
    print('is a property of the data rather than of any model we have tried.')
elif best_centre < 0.0 or best_centre > 0.45:
    print(f'Peak sits at {best_centre * 1000:.0f} ms, outside the plausible window for')
    print('a visual evoked response. If late, the')
    print('marker stream lags the stimulus and every epoch so far has been cut from the')
    print('wrong part of the trace. Re-run the features notebook with TMIN shifted.')
else:
    print(f'Peak at {best_centre * 1000:.0f} ms is where a visual response belongs, so')
    print('the epoching is right and the signal is real but weak.')
    print('That points back at the model, and the uncapped wd 1e0 run is owed.')